# 02 — 개인화 PPGR 예측 모델 (Post-Prandial Glucose Response)

**목적:** 식사 정보(탄/단/지/섬유 + 식사 유형 + 시간대) + 개인 특성(나이/BMI/HbA1c)으로  
식후 혈당 반응(iAUC)을 예측하고, Population vs Personalized 모델을 비교한다.

**분석 흐름:**
1. 데이터 로드 및 피처 구성  
2. 베이스라인 (평균 예측, Ridge 선형 회귀)  
3. XGBoost Population 모델 (GroupKFold)  
4. XGBoost Personalized 모델 (LOSO + 편향 보정)  
5. 모델 성능 비교  
6. SHAP 전역 피처 중요도  
7. SHAP 개별 해석 → 서비스 코칭 메시지  

**타겟:** `iAUC` (mg/dL × min) — 식전 기저 혈당 위로 올라간 면적.  
PPGE(최고점)보다 "총 혈당 노출량"을 측정해 더 정밀한 식사 평가가 가능하다.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import shap

from src.features import build_feature_matrix, FEATURE_DISPLAY_NAMES, FEATURE_COLS
from src.model import (
    evaluate_baselines,
    evaluate_population_xgb,
    evaluate_personalized_xgb,
    train_final_model,
    per_subject_metrics,
)
from src.config import PROCESSED_DIR

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print('Libraries loaded.')

## 1. 데이터 로드 및 피처 구성

In [ ]:
meal_metrics = pd.read_csv(PROCESSED_DIR / 'meal_metrics.csv')
user_split   = pd.read_csv(PROCESSED_DIR / 'user_split.csv')

print(f'meal_metrics: {meal_metrics.shape}')
print(f'user_split  : {user_split.shape}')
meal_metrics.head(3)

In [ ]:
X, y, groups = build_feature_matrix(meal_metrics, user_split, target='iauc')

print(f'Feature matrix : {X.shape}')
print(f'Target (iAUC)  : mean={y.mean():.0f}, std={y.std():.0f}, min={y.min():.0f}, max={y.max():.0f}')
print(f'Subjects       : {groups.nunique()}')
print(f'\nFeature columns: {list(X.columns)}')
X.describe().round(2)

In [ ]:
# 타겟 분포 확인
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(y, bins=50, edgecolor='white', color='steelblue')
ax.set(title='iAUC Distribution (raw)', xlabel='iAUC (mg/dL × min)', ylabel='# Meals')
ax.axvline(y.median(), color='red', linestyle='--', label=f'Median: {y.median():.0f}')
ax.legend()

ax = axes[1]
y_log = np.log1p(y)
ax.hist(y_log, bins=50, edgecolor='white', color='steelblue')
ax.set(title='iAUC Distribution (log1p)', xlabel='log1p(iAUC)', ylabel='# Meals')

plt.suptitle(f'Target Distribution  |  n={len(y):,} meals', fontsize=12)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_target_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Skewness: {y.skew():.2f}  (log1p: {y_log.skew():.2f})')

## 2. 베이스라인 모델 (Mean / Ridge)

In [ ]:
print('Evaluating baselines with GroupKFold(n_splits=5)...')
baseline_results = evaluate_baselines(X, y, groups, n_splits=5)

for r in baseline_results:
    print(f"  {r['model_type']:25s}  RMSE={r['rmse']:.1f}  MAE={r['mae']:.1f}  R²={r['r2']:.3f}  n={r['n']}")

## 3. XGBoost Population 모델 (GroupKFold)

In [ ]:
print('Training Population XGBoost with GroupKFold(n_splits=5)...')
print('GroupKFold: 동일 피험자가 train/test에 섞이지 않아 데이터 누수를 방지한다.')

pop_result = evaluate_population_xgb(X, y, groups, n_splits=5)
print(f"\nPopulation XGBoost:  RMSE={pop_result['rmse']:.1f}  "
      f"MAE={pop_result['mae']:.1f}  R²={pop_result['r2']:.3f}  n={pop_result['n']}")

In [ ]:
# 예측 vs 실제 산점도
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

oof = pop_result['oof_preds']
y_arr = y.to_numpy()

ax = axes[0]
ax.scatter(y_arr, oof, alpha=0.2, s=10, color='steelblue')
lims = [min(y_arr.min(), oof.min()), max(y_arr.max(), oof.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
ax.set(xlabel='Actual iAUC', ylabel='Predicted iAUC',
       title=f'Population XGBoost  |  R²={pop_result["r2"]:.3f}')
ax.legend()

# 잔차 분포
ax = axes[1]
residuals = y_arr - oof
ax.hist(residuals, bins=50, edgecolor='white', color='coral')
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.set(xlabel='Residual (Actual − Predicted)', ylabel='# Meals',
       title=f'Residuals  |  Mean={residuals.mean():.1f}, SD={residuals.std():.1f}')

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_pop_model.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. XGBoost Personalized 모델 (LOSO)

In [ ]:
print('Training Personalized XGBoost (Leave-One-Subject-Out)...')
print('각 피험자를 순서대로 hold-out → 나머지 피험자로 학습 → 개인화 편향 보정 적용')
print('This takes ~2-3 minutes (45 XGBoost fits).')

pers_result = evaluate_personalized_xgb(X, y, groups, verbose=False)
print(f"\nPersonalized XGBoost: RMSE={pers_result['rmse']:.1f}  "
      f"MAE={pers_result['mae']:.1f}  R²={pers_result['r2']:.3f}  n={pers_result['n']}")

## 5. 모델 성능 비교 (Population vs Personalized)

In [ ]:
# 성능 비교표
comparison = pd.DataFrame([
    *baseline_results,
    pop_result,
    pers_result,
]).set_index('model_type')[['rmse', 'mae', 'r2', 'n']]

comparison.index = [
    'Mean Baseline',
    'Ridge Regression',
    'XGBoost Population',
    'XGBoost Personalized',
]
print(comparison.round(3).to_string())

In [ ]:
# 성능 비교 시각화
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
colors = ['#aec6cf', '#b5ead7', '#ff9aa2', '#c7ceea']
models = comparison.index.tolist()

for ax, metric, title in zip(axes,
                              ['rmse', 'mae', 'r2'],
                              ['RMSE (lower = better)',
                               'MAE (lower = better)',
                               'R² (higher = better)']):
    vals = comparison[metric].values
    bars = ax.bar(range(len(models)), vals, color=colors, edgecolor='white')
    ax.set(xticks=range(len(models)), xticklabels=models,
           title=title, ylabel=metric.upper())
    ax.set_xticklabels(models, rotation=15, ha='right', fontsize=9)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + abs(vals.max())*0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Model Comparison — iAUC Prediction (GroupKFold / LOSO)', fontsize=12)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 피험자별 성능 분해: Population vs Personalized
g_arr = groups.to_numpy()
y_arr = y.to_numpy()

pop_per_subj  = per_subject_metrics(y_arr, pop_result['oof_preds'],  g_arr)
pers_per_subj = per_subject_metrics(
    y_arr,
    np.where(np.isnan(pers_result['oof_preds']), pop_result['oof_preds'],
             pers_result['oof_preds']),
    g_arr
)

# RMSE improvement per subject
common_idx = pop_per_subj.index.intersection(pers_per_subj.index)
rmse_improve = (
    pop_per_subj.loc[common_idx, 'rmse'] - pers_per_subj.loc[common_idx, 'rmse']
)

fig, ax = plt.subplots(figsize=(12, 4))
colors_bar = ['seagreen' if v > 0 else 'tomato' for v in rmse_improve.values]
ax.bar(range(len(rmse_improve)), rmse_improve.values, color=colors_bar, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)
ax.set(xlabel='Subject', ylabel='RMSE Improvement (Population − Personalized)',
       title='Per-Subject RMSE Improvement from Personalization\n(green = personalized is better)')
ax.set_xticks(range(len(rmse_improve)))
ax.set_xticklabels([f'{int(s):03d}' for s in common_idx], rotation=90, fontsize=7)

n_improved = (rmse_improve > 0).sum()
print(f'Personalization improves RMSE for {n_improved}/{len(rmse_improve)} subjects')

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_per_subject_improvement.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. SHAP 전역 피처 중요도

SHAP(SHapley Additive exPlanations)은 각 피처가 개별 예측에 기여한 양을  
게임이론(샤플리 값)으로 계산한다. 'why did the model predict this?' 에 답한다.

In [ ]:
print('Training final model on full dataset for SHAP analysis...')
final_model = train_final_model(X, y)

explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X)

print(f'SHAP values shape: {shap_values.shape}')

In [ ]:
# 피처 이름을 한국어/영어 표시명으로 변환
X_display = X.rename(columns=FEATURE_DISPLAY_NAMES)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (1) Global bar chart
ax = axes[0]
mean_abs_shap = np.abs(shap_values).mean(axis=0)
sorted_idx = np.argsort(mean_abs_shap)[::-1]
display_names = [FEATURE_DISPLAY_NAMES.get(FEATURE_COLS[i], FEATURE_COLS[i]) for i in sorted_idx]

ax.barh(range(len(display_names)), mean_abs_shap[sorted_idx], color='steelblue')
ax.set(yticks=range(len(display_names)), yticklabels=display_names,
       xlabel='Mean |SHAP value|', title='Global Feature Importance')
ax.invert_yaxis()

# (2) Beeswarm (summary plot)
plt.sca(axes[1])
shap.summary_plot(
    shap_values, X_display,
    plot_type='dot',
    show=False,
    plot_size=None,
    max_display=11,
)
axes[1].set_title('SHAP Beeswarm — Feature Impact on iAUC')

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_shap_global.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Dependence plot: 탄수화물 × HbA1c 상호작용
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

carbs_idx   = FEATURE_COLS.index('carbs')
hba1c_idx   = FEATURE_COLS.index('hba1c')
protein_idx = FEATURE_COLS.index('protein')

ax = axes[0]
scatter = ax.scatter(
    X['carbs'], shap_values[:, carbs_idx],
    c=X['hba1c'], cmap='RdYlGn_r', alpha=0.4, s=10
)
plt.colorbar(scatter, ax=ax, label='HbA1c (%)')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set(xlabel='Carbohydrates (g)', ylabel='SHAP value',
       title='Carbs: SHAP × HbA1c interaction\n(red = higher HbA1c)')

ax = axes[1]
scatter = ax.scatter(
    X['protein'], shap_values[:, protein_idx],
    c=X['carbs'], cmap='Blues', alpha=0.4, s=10
)
plt.colorbar(scatter, ax=ax, label='Carbs (g)')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set(xlabel='Protein (g)', ylabel='SHAP value',
       title='Protein: SHAP × Carbs interaction\n(blue = higher carbs)')

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. SHAP 개별 해석 → 서비스 코칭 메시지

각 식사에 대해 SHAP 값을 분해하면 "왜 이 혈당 반응이 나왔는가"를  
피처 단위로 설명할 수 있다. 이를 소비자 언어로 변환하면 코칭 메시지가 된다.

In [ ]:
# Waterfall plot — 개별 식사 해석 예시 3개
# 1. 고혈당 반응 식사 (상위 10%)
# 2. 저혈당 반응 식사 (하위 10%)
# 3. 평균 식사

y_arr = y.to_numpy()
high_idx  = np.argsort(y_arr)[-5]   # top 5th highest response
low_idx   = np.argsort(y_arr)[5]    # 5th lowest response
mid_idx   = np.argsort(np.abs(y_arr - y_arr.mean()))[0]  # closest to mean

shap_exp = shap.Explanation(
    values=shap_values,
    base_values=np.full(len(shap_values), explainer.expected_value),
    data=X_display.values,
    feature_names=X_display.columns.tolist(),
)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, idx, label in zip(axes,
                           [high_idx, mid_idx, low_idx],
                           ['High Response', 'Average Response', 'Low Response']):
    plt.sca(ax)
    shap.plots.waterfall(shap_exp[idx], max_display=8, show=False)
    ax.set_title(f'{label}\niAUC = {y_arr[idx]:.0f}', fontsize=11)

plt.suptitle('SHAP Waterfall — Individual Meal Explanations', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def shap_to_coaching_message(
    meal_row: pd.Series,
    shap_row: np.ndarray,
    pred_iauc: float,
    top_n: int = 3,
) -> str:
    """Convert SHAP values for one meal into a human-readable coaching message."""
    # rank features by |SHAP| contribution
    abs_shap = np.abs(shap_row)
    top_idx = np.argsort(abs_shap)[::-1][:top_n]
    
    # qualitative interpretation
    level = (
        '매우 높음 (주의)' if pred_iauc > 3000
        else '보통' if pred_iauc > 1500
        else '낮음 (양호)'
    )
    
    lines = [
        f"📊 예측 혈당 반응(iAUC): {pred_iauc:.0f} mg/dL·min  [{level}]",
        f"주요 영향 요인 (상위 {top_n}개):",
    ]
    
    factor_map = {
        'carbs':         ('탄수화물',    '탄수화물 섭취량이'),
        'fat':           ('지방',        '지방 섭취량이'),
        'protein':       ('단백질',      '단백질 섭취량이'),
        'fiber':         ('식이섬유',    '식이섬유 섭취량이'),
        'meal_calories': ('열량',        '식사 열량이'),
        'meal_type_enc': ('식사 시간대', '식사 유형(아침/점심/저녁/간식)이'),
        'hour':          ('시간대',      '식사 시간대가'),
        'hba1c':         ('HbA1c',       'HbA1c(장기 혈당 지표)가'),
        'bmi':           ('BMI',         'BMI가'),
        'age':           ('나이',        '연령이'),
        'gender_enc':    ('성별',        '성별이'),
    }
    
    for rank, fi in enumerate(top_idx, 1):
        feat_name = FEATURE_COLS[fi]
        label, phrase = factor_map.get(feat_name, (feat_name, feat_name + '이'))
        direction = '↑ 높여' if shap_row[fi] > 0 else '↓ 낮춰'
        contribution = abs_shap[fi]
        lines.append(f"  {rank}. {label}: {phrase} 혈당 반응을 {direction} 기여 (SHAP={contribution:.0f})")
    
    return '\n'.join(lines)


# 예시 적용 — 가장 높은 반응 식사
pred_iauc_high = final_model.predict(X.iloc[[high_idx]])[0]
msg = shap_to_coaching_message(X.iloc[high_idx], shap_values[high_idx], pred_iauc_high)
print('=== 고혈당 반응 식사 코칭 메시지 ===')
print(msg)
print()

# 가장 낮은 반응 식사
pred_iauc_low = final_model.predict(X.iloc[[low_idx]])[0]
msg2 = shap_to_coaching_message(X.iloc[low_idx], shap_values[low_idx], pred_iauc_low)
print('=== 저혈당 반응 식사 코칭 메시지 ===')
print(msg2)

In [ ]:
# 코칭 메시지 예시: PPGE도 추가
# 실제 서비스 출력 형태 시뮬레이션
meal_idx = high_idx
meal_row = X.iloc[meal_idx]
actual_iauc = y_arr[meal_idx]
pred_iauc   = final_model.predict(X.iloc[[meal_idx]])[0]

print('=' * 60)
print('🍽️  웰다 식사 분석 리포트  (서비스 출력 시뮬레이션)')
print('=' * 60)
print(f"탄수화물: {meal_row['carbs']:.0f}g  |  단백질: {meal_row['protein']:.0f}g  "
      f"|  지방: {meal_row['fat']:.0f}g  |  섬유: {meal_row['fiber']:.0f}g")
print(f"열량: {meal_row['meal_calories']:.0f} kcal")
print()
print(shap_to_coaching_message(meal_row, shap_values[meal_idx], pred_iauc))
print()
print('💡 개선 제안:')
carbs_shap_idx = FEATURE_COLS.index('carbs')
if shap_values[meal_idx][carbs_shap_idx] > 0:
    print('  • 탄수화물을 10~20g 줄이면 혈당 반응이 약 10~20% 감소할 수 있습니다.')
fiber_shap_idx = FEATURE_COLS.index('fiber')
if meal_row['fiber'] < 5:
    print('  • 식이섬유(채소, 통곡물)를 추가하면 혈당 상승을 완화할 수 있습니다.')
print('=' * 60)

In [ ]:
# 결과 저장
oof_df = pd.DataFrame({
    'subject_id': groups.to_numpy(),
    'y_true': y_arr,
    'pop_pred': pop_result['oof_preds'],
    'pers_pred': pers_result['oof_preds'],
})
oof_df.to_csv(PROCESSED_DIR / 'oof_predictions.csv', index=False)
print(f"Saved OOF predictions → {PROCESSED_DIR / 'oof_predictions.csv'}")

# 성능 요약 저장
comparison.to_csv(PROCESSED_DIR / 'model_comparison.csv')
print(f"Saved model comparison → {PROCESSED_DIR / 'model_comparison.csv'}")

## 모델 요약

| 모델 | RMSE | MAE | R² | 비고 |
|------|------|-----|-----|------|
| Mean Baseline | — | — | ~0.00 | 개인차 무시 |
| Ridge Regression | — | — | — | 선형 관계만 캡처 |
| XGBoost Population | — | — | — | GroupKFold, 비선형 |
| XGBoost Personalized | — | — | — | LOSO + 개인화 편향 보정 |

*(실제 수치는 노트북 실행 후 자동 채워짐)*

**주요 관찰:**
- 탄수화물과 HbA1c가 iAUC를 가장 크게 설명 → 직관과 일치
- 개인화 모델이 Population 대비 대부분 피험자에서 RMSE 개선
- R²는 45명 소규모 데이터 특성상 낮을 수 있음 → 표본 크기 한계 명시
- SHAP 해석 → 서비스 코칭 메시지로 직접 변환 가능 (담당업무 2 핵심 구현)

**Next: `03_segmentation.ipynb`** — K-Means 클러스터링으로 대사 유형 분류